<h1 style="text-align: center;">[Sinarmas Land]</h1>
<h3 style="text-align: center;">[Mikha, Hane, Nadya]</h3>

---

## **Section 1. Business Context**

**1.1 Context**

Sinarmas Land telah meredefinisi konsep pengembangan properti di Indonesia dengan tidak sekadar membangun perumahan, melainkan merancang Kota Mandiri terpadu (Township). Pendapatan ini bersumber dari Iuran Pemeliharaan Lingkungan (IPL), tagihan utilitas (air bersih kawasan), serta penyewaan area komersial.

**1.2 Problem Statements**

Membengkaknya defisit operasional Township Management akibat tingginya tunggakan IPL (Iuran Pemeliharaan Lingkungan) dari properti kosong serta kacaunya sistem pencatatan tagihan utilitas/air warga.

**1.3 Key Objective**

Revenue Recovery: Membangun dashboard aging schedule (umur piutang) untuk memfokuskan tim Collection mengejar pemilik rumah investasi yang menunggak miliaran rupiah.

Utility System Patching: Menambahkan fungsi validasi pada aplikasi petugas catat meteran, sehingga input pemakaian air yang melonjak >300% dari bulan sebelumnya akan otomatis diblokir untuk verifikasi ulang.

Data Cleansing: Menstandarisasi penamaan metode pembayaran agar tim rekonsiliasi Finance tidak perlu begadang tiap akhir bulan mencocokkan mutasi rekening bank.

Anti-Fraud/Bug Fix: Menutup bug logika ERP yang menerbitkan dan menerima pembayaran invoice untuk unit yang tanahnya saja bahkan belum dibangun.


## **Section 2. Data Understanding**

# 2.1 General Information

**Skala Dataset**

Dataset terdiri dari 3 tabel utama dengan total sekitar 300.000 baris data tagihan bulanan.

sml_clusters.csv → dimensi cluster/kawasan.

sml_units.csv → dimensi unit rumah/ruko.

sml_ipl_billings.csv → fakta tagihan bulanan.

**Tujuan Pengumpulan Data**

Data ini dikumpulkan oleh Estate Management Sinarmas Land untuk mendukung operasional township (BSD City, Kota Wisata, Grand Wisata, NavaPark). Fokusnya adalah:

Penagihan IPL (Iuran Pemeliharaan Lingkungan).

Tagihan utilitas (air bersih).

Monitoring kepatuhan pembayaran warga.

**Karakteristik Data**

Granularitas: data bulanan per unit rumah.

Relasi antar tabel:

cluster_id menghubungkan unit dengan cluster.

unit_id menghubungkan unit dengan tagihan bulanan.

Variasi data: mencakup informasi demografi (pemilik, kontak), status rumah (vacant/occupied), serta transaksi finansial (invoice, payment).

**Potensi Insight**

Analisis aging schedule piutang (tunggakan IPL).

Identifikasi cluster dengan rasio tunggakan tinggi.

Validasi kualitas data (missing values, outliers, logical errors).

Deteksi fraud/anomali pada sistem ERP.

**2.2 Feature Information**

| **Dataset** | **Kolom** | **Tipe Data** | **Deskripsi** | **Relevansi Analisis** |
| --- | --- | --- | --- | --- |
| **sml_clusters.csv** | ``cluster_id`` | Integer/String | ID unik cluster | Identifikasi cluster/township untuk analisis rasio tunggakan |
|  | ``township_name`` | String | Nama kota mandiri (BSD City, Kota Wisata, dll) | Segmentasi berdasarkan township |
|  | ``cluster_category`` | Categorical String | Segmen pasar (Premium, Middle, Commercial) | Analisis perbandingan antar kategori pasar |
| **sml_units.csv** | ``unit_id`` | Integer/String | ID unik rumah/ruko | Relasi ke tagihan bulanan |
|  | ``cluster_id`` | Foreign Key | Relasi ke ``sml_clusters`` | Hubungan unit dengan cluster |
|  | ``owner_name`` | String | Nama pemilik unit | Identifikasi pemilik rumah |
|  | ``contact_number`` | String | Nomor telepon pemilik (sering kosong) | Penting untuk tim Collection, rawan missing values |
|  | ``is_vacant`` | Boolean | Status rumah kosong (True/False) | Analisis rumah investasi kosong |
|  | ``handover_date`` | Datetime | Tanggal serah terima unit | Validasi logical error (payment sebelum handover) |
| **sml_ipl_billings.csv** | ``invoice_id`` | Integer/String | Nomor tagihan unik | Identifikasi invoice |
|  | ``unit_id`` | Foreign Key | Relasi ke ``sml_units`` | Hubungan tagihan dengan unit |
|  | ``billing_month`` | Period (YYYY-MM) | Periode bulan tagihan | Analisis aging schedule piutang |
|  | ``water_usage_m3`` | Numeric (Float/Int) | Pemakaian air dalam m³ | Deteksi outlier ekstrem |
|  | ``ipl_fee`` | Numeric | Biaya standar IPL | Komponen biaya tetap |
|  | ``total_amount`` | Numeric | Total tagihan (IPL + Air + Denda) | Estimasi revenue dan tunggakan |
|  | ``payment_status`` | Categorical String | Status pembayaran (Paid, Unpaid, Overdue) | Analisis kepatuhan pembayaran |
|  | ``payment_method`` | Categorical String | Metode pembayaran (BCA VA, Tokopedia, Cash, dll) | Evaluasi standar pencatatan metode pembayaran |
|  | ``payment_date`` | Datetime | Tanggal pembayaran | Validasi logical error (Paid sebelum handover) |

**2.3 Statistics Summary**

| **Aspek** | **Temuan Awal (Estimasi/Kualitatif)** | **Relevansi Analisis** |
| --- | --- | --- |
| **Jumlah Data** | ± 300.000 baris tagihan bulanan, puluhan ribu unit rumah, ratusan cluster | Skala besar → butuh data cleaning & validasi sistematis |
| **Rumah Kosong (is_vacant)** | Proporsi signifikan unit berstatus *vacant* (investasi) | Kontributor utama tunggakan IPL |
| **Payment Status** | Banyak invoice berstatus *Unpaid* atau *Overdue* | Fokus untuk aging schedule piutang & revenue recovery |
| **Payment Method** | Tidak konsisten: “BCA VA”, “VA BCA”, “Tokped”, “Tokopedia”, “Cash”, dll | Menyulitkan rekonsiliasi Finance, perlu standardisasi |
| **Contact Number** | Persentase missing values tinggi (kolom kosong) | Hambatan bagi tim Collection dalam menagih tunggakan |
| **Water Usage (m³)** | Outlier ekstrem (misal >10.000 m³ untuk rumah kecil) | Memicu komplain salah tagih, perlu anomaly detection |
| **Logical Error (Payment Date)** | Ada kasus *Paid* sebelum *handover_date* unit | Bug ERP → indikasi fraud/ketidaksesuaian sistem |
| **Total Amount** | Variasi besar, termasuk nilai minus atau melonjak tidak wajar | Perlu validasi formula perhitungan tagihan |

## **Section 3. Data Cleaning**

**3.1 Missing Values**

**3.2 Duplicated Values**

**3.3 Identify Spelling Errors**

**3.4 Identify Anomaly Values**

## **Section 4. Analytics**

### **4.1 Question 1**

### **4.2 Question 2**

## **Section 5. Conclusion and Recommendation**

**5.1 Conclusion**

**5.2 Recommendation**